# Phase 6 — Architecture 4: LegalBlock-GQA

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','-q','install','bert-score','rouge-score'])
from pathlib import Path
import os, json, math, random, sys, pickle, subprocess, zipfile, time
from collections import Counter, defaultdict
import cloudpickle, torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download

ARCH = 4  # LegalBlock-GQA: RoPE, RMSNorm, SwiGLU, grouped-query attention
BASE = Path('/kaggle/input/datasets/raaz74/a4-p4')
WEIGHTS, BPE = BASE / 'best.pt', BASE / 'tokenizer.json'
OUT = Path('/kaggle/working/phase6_a4_final')
OUT.mkdir(exist_ok=True)
METHODS = ['lora', 'lorma', 'pfeiffer', 'houlsby']
SOURCE, TARGET, BATCH = 1024, 640, 1
EPOCHS, PATIENCE, EVAL_STEPS = 8, 5, 250
SEED = 781
tok = Tokenizer.from_file(str(BPE))
PAD = tok.token_to_id('<pad>')
BOS = tok.token_to_id('<bos>')
EOS = tok.token_to_id('<eos>')
VOCAB = tok.get_vocab_size()

In [2]:
def read_rows(split):
    path = hf_hub_download('Exploration-Lab/CS-781-Capstone',
        f'Phase_6_Summ/{split.title()}/{split}.json', repo_type='dataset',
        token=os.environ.get('HF_TOKEN'))
    rows = json.loads(Path(path).read_text())
    if isinstance(rows, dict):
        rows = [dict(v, id=v.get('id', k)) if isinstance(v, dict) else {'id': k, 'document': v}
                for k, v in rows.items()]
    return rows

def prepare(split):
    data = []
    for row in read_rows(split):
        doc = tok.encode(row['document'], add_special_tokens=False).ids
        budget = SOURCE - 1
        if len(doc) > budget:
            quarter = budget // 4
            starts = [0, (len(doc)-quarter)//3, 2*(len(doc)-quarter)//3]
            doc = sum((doc[i:i+quarter] for i in starts), []) + doc[-(budget-3*quarter):]
        target = []
        if split == 'train':
            target = tok.encode(row['summary'], add_special_tokens=False).ids
            if len(target) < TARGET:
                target.append(EOS)
            else:
                target = target[:TARGET]
        data.append((row['id'], doc + [BOS], target))
    return data

rows = prepare('train')
ids = sorted({str(r[0]) for r in rows})
random.Random(SEED).shuffle(ids)
held = set(ids[:max(1, round(.1*len(ids)))])
train = [r for r in rows if str(r[0]) not in held]
dev = [r for r in rows if str(r[0]) in held]
test = prepare('test')
dev_sample = random.Random(SEED).sample(dev, min(32,len(dev)))
wanted = {str(r[0]) for r in dev_sample}
refs = {str(r['id']):r['summary'] for r in read_rows('train') if str(r['id']) in wanted}
with (OUT/'calibration.pkl').open('wb') as f:
    pickle.dump((dev_sample,refs),f)
with (OUT/'data.pkl').open('wb') as f:
    pickle.dump((train,dev,test), f)
print(f'{len(train):,} train | {len(dev):,} dev | {len(test):,} test')

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

6,327 train | 703 dev | 100 test


In [3]:
# LegalBlock-GQA: RoPE, RMSNorm, SwiGLU, grouped-query attention
class RMSNorm(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return (x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + 1e-6)).to(x.dtype) * self.weight

class Attention(nn.Module):
    def __init__(self, c, global_attn):
        super().__init__()
        self.h, self.kv, self.d = c['heads'], c['kv_heads'], c['dim'] // c['heads']
        self.window, self.global_attn = c['window'], global_attn
        self.qkv = nn.Linear(c['dim'], (self.h + 2*self.kv)*self.d, bias=False)
        self.out = nn.Linear(c['dim'], c['dim'], bias=False)
        self.register_buffer('freq', 1 / (10000 ** (torch.arange(0, self.d, 2).float()/self.d)), persistent=False)
    def rope(self, x, pos):
        angle = (torch.arange(pos, pos+x.size(2), device=x.device).float()[:, None] * self.freq).to(x.dtype)
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a*angle.cos()-b*angle.sin(), a*angle.sin()+b*angle.cos()), -1).flatten(-2)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        b, t, _ = x.shape
        q, k, v = self.qkv(x).split([self.h*self.d, self.kv*self.d, self.kv*self.d], -1)
        q = self.rope(q.view(b,t,self.h,self.d).transpose(1,2), pos)
        k = self.rope(k.view(b,t,self.kv,self.d).transpose(1,2), pos)
        v = v.view(b,t,self.kv,self.d).transpose(1,2)
        if cache is not None and (self.global_attn or pos % self.window != 0):
            k, v = torch.cat((cache[0], k), 2), torch.cat((cache[1], v), 2)
        saved = None
        if use_cache:
            start = 0 if self.global_attn else ((pos+t-1)//self.window)*self.window - pos
            start = max(0, start) if cache is None else 0
            saved = (k[:,:,start:], v[:,:,start:])
        k, v = k.repeat_interleave(self.h//self.kv, 1), v.repeat_interleave(self.h//self.kv, 1)
        if t == 1:
            y = F.scaled_dot_product_attention(q, k, v)
        elif self.global_attn:
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            w = self.window
            pad = (-t) % w
            q, k, v = [F.pad(z, (0,0,0,pad)).reshape(b,self.h,-1,w,self.d)
                       .permute(0,2,1,3,4).reshape(-1,self.h,w,self.d) for z in (q,k,v)]
            y = F.scaled_dot_product_attention(q,k,v,is_causal=True)
            y = y.reshape(b,-1,self.h,w,self.d).permute(0,2,1,3,4).reshape(b,self.h,-1,self.d)[:,:,:t]
        return self.out(y.transpose(1,2).reshape(b,t,-1)), saved

class Bottleneck(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.down = nn.Linear(width, 32)
        self.up = nn.Linear(32, width)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = x + self.up(F.relu(self.down(x)))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x

class Block(nn.Module):
    def __init__(self, c, i):
        super().__init__()
        self.n1, self.n2 = RMSNorm(c['dim']), RMSNorm(c['dim'])
        self.attn = Attention(c, (i+1) % c['global_every'] == 0)
        self.up = nn.Linear(c['dim'], 2*c['ff'], bias=False)
        self.down = nn.Linear(c['ff'], c['dim'], bias=False)
    def forward(self, x, cache=None, pos=0, use_cache=False):
        y, cache = self.attn(self.n1(x), cache, pos, use_cache)
        x = x + y
        if hasattr(self, "att_adapter"): x = self.att_adapter(x)
        a, b = self.up(self.n2(x)).chunk(2, -1)
        x = x + self.down(F.silu(a)*b)
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x, cache

class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.emb = nn.Embedding(c['vocab'], c['dim'])
        self.blocks = nn.ModuleList([Block(c, i) for i in range(c['layers'])])
        self.norm = RMSNorm(c['dim'])
        self.apply(self.init_weights)
        for block in self.blocks:
            nn.init.normal_(block.attn.out.weight, std=.02/math.sqrt(2*c['layers']))
            nn.init.normal_(block.down.weight, std=.02/math.sqrt(2*c['layers']))
    def init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=.02)
    def forward(self, ids, targets=None, cache=None, pos=0, use_cache=False):
        x, saved = self.emb(ids), []
        for i, block in enumerate(self.blocks):
            if self.training:
                x, _ = checkpoint(block, x, use_reentrant=False)
            else:
                x, kv = block(x, None if cache is None else cache[i], pos, use_cache)
                saved.append(kv)
        x = self.norm(x)
        if targets is None:
            return F.linear(x[:,-1], self.emb.weight), saved
        logits = F.linear(x, self.emb.weight)
        return F.cross_entropy(logits.flatten(0,1), targets.flatten(), ignore_index=-100)

In [4]:
class Adapt(nn.Module):
    def __init__(self, base, method):
        super().__init__()
        self.base, self.method = base, method
        if method == 'scale':
            self.scale = nn.Parameter(torch.ones(base.out_features))
        else:
            width = base.out_features if method == 'lorma' else base.in_features
            self.A = nn.Linear(width, 8, bias=False)
            self.B = nn.Linear(8, base.out_features, bias=False)
            nn.init.zeros_(self.B.weight)

    def forward(self, x):
        y = self.base(x)
        if self.method == 'scale':
            return y * self.scale.to(y.dtype)
        return y + self.B(self.A(y if self.method == 'lorma' else x))

In [5]:
def build(method):
    ck = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
    cfg = dict(ck['config'], gradient_checkpointing=False, pad_id=PAD)
    m = LegalLM(cfg)
    m.load_state_dict(ck['model'])
    m.requires_grad_(False)
    blocks = m.blocks
    width = cfg.get('dim',cfg.get('width'))
    if method in ('houlsby','pfeiffer'):
        for b in blocks:
            b.ff_adapter = Bottleneck(width)
            if method == 'houlsby': b.att_adapter = Bottleneck(width)
    else:
        for parent in list(m.modules()):
            for name, layer in list(parent.named_children()):
                if isinstance(layer, nn.Linear) and name in ('qkv','q','kv','out'):
                    setattr(parent, name, Adapt(layer, method))
    return m

In [6]:
def logits(m, batch):
    inp = pad([r[1]+r[2][:-1] for r in batch], PAD)
    lens = torch.tensor([len(r[1]) for r in batch], device=inp.device)
    y = pad([[-100]*(len(r[1])-1)+r[2] for r in batch], -100)
    x = m.emb(inp)
    for b in m.blocks:
        x = checkpoint(lambda h, block=b: block(h)[0], x, use_reentrant=False) if m.training else b(x)[0]
    x = m.norm(x)
    keep = y != -100
    return F.linear(x[keep], m.emb.weight), y[keep]

In [7]:
def pad(rows, value):
    return nn.utils.rnn.pad_sequence([torch.tensor(r) for r in rows], batch_first=True,
                                     padding_value=value).cuda()

def batches(rows, seed=None):
    order = list(range(len(rows)))
    if seed is not None: random.Random(seed).shuffle(order)
    for start in range(0, len(order), 64):
        chunk = sorted(order[start:start+64], key=lambda i: len(rows[i][1])+len(rows[i][2]))
        for j in range(0,len(chunk),BATCH):
            yield [rows[i] for i in chunk[j:j+BATCH]]

def load_best(method):
    m = build(method)
    m.load_state_dict(torch.load(OUT/method/'best.pt',map_location='cpu',weights_only=True),strict=False)
    return m.cuda().eval()

def train_one(method):
    torch.manual_seed(SEED)
    with (OUT/'data.pkl').open('rb') as f: train, dev, _ = pickle.load(f)
    dest = OUT/method
    dest.mkdir(exist_ok=True)
    m = build(method).cuda()
    params = {k:p for k,p in m.named_parameters() if p.requires_grad}
    teacher = load_best('lora') if method == 'distill' else None
    lr = 2e-5 if method in ('partial','decoder','cross','distill') else 3e-4
    opt = torch.optim.AdamW(params.values(),lr=lr,fused=True)
    scaler = torch.amp.GradScaler('cuda')
    best, stale, step = float('inf'), 0, 0
    start = time.monotonic()
    print(method, 'trainable', sum(p.numel() for p in params.values()), flush=True)
    for epoch in range(EPOCHS):
        for j, batch in enumerate(batches(train, SEED+epoch), 1):
            m.train()
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                z,y = logits(m,batch)
                loss = F.cross_entropy(z.float(),y)
                if teacher is not None:
                    with torch.no_grad(): t,_ = logits(teacher,batch)
                    kd = F.kl_div((z.float()/2).log_softmax(-1),(t.float()/2).softmax(-1),reduction='batchmean')
                    loss = .5*loss + 2*kd
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params.values(),1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step % EVAL_STEPS and j != math.ceil(len(train)/BATCH): continue
            m.eval()
            total, count = torch.zeros((), device='cuda'), 0
            with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16):
                for v in batches(dev):
                    z,y = logits(m,v)
                    total += F.cross_entropy(z.float(),y,reduction='sum')
                    count += len(y)
            score = total.item()/count
            print(method, 'epoch',epoch+1,'step',step,'dev NLL',round(score,4),flush=True)
            if score < best:
                best, stale = score, 0
                torch.save({k:p.detach().half().cpu() for k,p in params.items()},dest/'best.tmp')
                os.replace(dest/'best.tmp',dest/'best.pt')
            else: stale += 1
            if stale >= PATIENCE: break
        if stale >= PATIENCE: break
    (dest/'done').write_text(json.dumps(dict(dev_nll=best,steps=step,seconds=time.monotonic()-start,
                    peak_GiB=torch.cuda.max_memory_allocated()/2**30)))

In [8]:
@torch.inference_mode()
@torch.autocast('cuda',dtype=torch.float16)
def generate(m,doc,limit=TARGET):
    ids = torch.tensor([doc],device='cuda')
    z,cache = m(ids,use_cache=True)
    result, followers = [], defaultdict(set)
    for i in range(limit):
        z = z[:, :VOCAB].float()
        for special in (PAD,BOS):
            if special != EOS: z[:,special] = -torch.inf
        if i < 8: z[:,EOS] = -torch.inf
        if len(result)>=2:
            z[:,list(followers[tuple(result[-2:])])] = -torch.inf
        token = z.argmax(-1)
        n = token.item()
        if n == EOS: break
        if len(result)>=2: followers[tuple(result[-2:])].add(n)
        result.append(n)
        z,cache = m(token[:,None],cache=cache,pos=ids.size(1)+i,use_cache=True)
    return result

def predict(method):
    m = load_best(method)
    tokenizer = Tokenizer.from_file(str(BPE))
    with (OUT/'data.pkl').open('rb') as f: _,_,test = pickle.load(f)
    from rouge_score.rouge_scorer import RougeScorer
    from bert_score import score as bertscore
    scorer = RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    with (OUT/'calibration.pkl').open('rb') as f: sample,refs = pickle.load(f)
    generated = [generate(m,d) for _,d,_ in sample]
    gold = [refs[str(k)] for k,_,_ in sample]
    caps = [192,384,TARGET]
    texts = [[tokenizer.decode(t[:cap]).strip() for t in generated] for cap in caps]
    _,_,f1 = bertscore(sum(texts,[]),gold*len(caps),lang='en',model_type='roberta-large',
                       batch_size=4,device='cuda',verbose=False)
    metrics = []
    for cap,group,bert in zip(caps,texts,f1.reshape(len(caps),-1).mean(1).tolist()):
        rs = [scorer.score(ref,hyp) for ref,hyp in zip(gold,group)]
        metrics.append(dict(cap=cap,bert=bert,**{k:sum(r[k].fmeasure for r in rs)/len(rs)
                                               for k in ('rouge1','rougeL')}))
    peak = max(r['bert'] for r in metrics)
    chosen = max((r for r in metrics if r['bert'] >= peak-.002),
                 key=lambda r:r['rouge1']+r['rougeL'])
    (OUT/method/'decoding.json').write_text(json.dumps(dict(selected=chosen,results=metrics),indent=2))
    print(method,chosen,flush=True)
    rows = [dict(id=k,summary=tokenizer.decode(generate(m,d,chosen["cap"])).strip()) for k,d,_ in test]
    package(method,rows)

def package(method,rows):
    with zipfile.ZipFile(OUT/f'submission_{method}.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',json.dumps(rows,ensure_ascii=False))

In [9]:
def run_jobs(names, worker):
    jobs = {}
    queue = list(names)
    payload = OUT/'worker.pkl'
    payload.write_bytes(cloudpickle.dumps(worker))
    command = "import cloudpickle,sys,torch;torch.set_num_threads(1);cloudpickle.load(open(sys.argv[1],'rb'))(sys.argv[2])"
    while queue or jobs:
        for gpu in (0, 1):
            if gpu in jobs:
                name, p = jobs[gpu]
                if p.poll() is None:
                    continue
                p.wait()
                del jobs[gpu]
            ready = [n for n in queue if worker != train_one or n != 'distill' or (OUT/'lora'/'done').exists()]
            if ready:
                name = ready[0]
                queue.remove(name)
                print(f'GPU {gpu} | {name}', flush=True)
                jobs[gpu] = name, subprocess.Popen(
                    [sys.executable, '-c', command, str(payload), name],
                    env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        if jobs:
            time.sleep(2)


In [10]:
run_jobs(METHODS, train_one)

GPU 0 | lora
GPU 1 | lorma
lorma trainable 884736
lora trainable 811008
lora epoch 1 step 250 dev NLL 2.3859
lorma epoch 1 step 250 dev NLL 2.3772
lora epoch 1 step 500 dev NLL 2.3526
lorma epoch 1 step 500 dev NLL 2.3505
lora epoch 1 step 750 dev NLL 2.3256
lorma epoch 1 step 750 dev NLL 2.3268
lora epoch 1 step 1000 dev NLL 2.3101
lorma epoch 1 step 1000 dev NLL 2.3096
lora epoch 1 step 1250 dev NLL 2.3046
lorma epoch 1 step 1250 dev NLL 2.3037
lora epoch 1 step 1500 dev NLL 2.2958
lorma epoch 1 step 1500 dev NLL 2.2975
lora epoch 1 step 1750 dev NLL 2.2876
lorma epoch 1 step 1750 dev NLL 2.2906
lora epoch 1 step 2000 dev NLL 2.2808
lorma epoch 1 step 2000 dev NLL 2.2861
lora epoch 1 step 2250 dev NLL 2.2742
lorma epoch 1 step 2250 dev NLL 2.2773
lora epoch 1 step 2500 dev NLL 2.2696
lorma epoch 1 step 2500 dev NLL 2.2732
lora epoch 1 step 2750 dev NLL 2.2671
lora epoch 1 step 3000 dev NLL 2.2632
lorma epoch 1 step 2750 dev NLL 2.2713
lora epoch 1 step 3250 dev NLL 2.2602
lorma epoch

In [11]:
run_jobs(METHODS, predict)

GPU 0 | lora
GPU 1 | lorma


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2590.20it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2503.45it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEX

lora {'cap': 640, 'bert': 0.8424869775772095, 'rouge1': 0.42127608563679964, 'rougeL': 0.22191013529222728}
lorma {'cap': 640, 'bert': 0.8413859605789185, 'rouge1': 0.416703195374061, 'rougeL': 0.21560302828958194}
GPU 0 | pfeiffer
GPU 1 | houlsby


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2549.22it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2755.02it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEX

pfeiffer {'cap': 384, 'bert': 0.8415318727493286, 'rouge1': 0.43360562746186176, 'rougeL': 0.21965237500317783}
houlsby {'cap': 640, 'bert': 0.8391781449317932, 'rouge1': 0.45140692449869624, 'rougeL': 0.22488106200875937}


In [12]:
def overlap(a,b):
    a,b = Counter(a.lower().split()),Counter(b.lower().split())
    return 2*sum((a&b).values())/max(1,sum(a.values())+sum(b.values()))

candidates = defaultdict(list)
for name in METHODS:
    with zipfile.ZipFile(OUT/f'submission_{name}.zip') as z:
        for row in json.loads(z.read('prediction.json')):
            candidates[row['id']].append(row['summary'])
ensemble = []
for key,_,_ in test:
    group = candidates[key]
    scores = [sum(overlap(s,t) for j,t in enumerate(group) if j != i) for i,s in enumerate(group)]
    ensemble.append(dict(id=key, summary=group[max(range(len(group)),key=scores.__getitem__)]))
package('ensemble',ensemble)
print(OUT)

/kaggle/working/phase6_a4_final


## Notes